# BTC Direction Classifier — MMT Order Flow (GPU, multi-test)

One notebook, every test driven by flags in section 0.

| test | USE_MMT | USE_TAPE | USE_ORDERBOOK | HORIZON | LABEL_MODE |
|---|---|---|---|---|---|
| 0 baseline (done) | False | - | - | 60 | close |
| 1 MMT 60m (done)  | True | False | True | 60 | close |
| 2 + tape speed    | True | True | True | 60 | close |
| 3 long window     | True | False | **False** | 60 | close |
| **4a horizon 1m** | True | True | True | **1** | close |
| **4b horizon 15m**| True | True | True | **15** | close |
| **5 barriers**    | True | True | True | (n/a) | **barrier** |

**Results so far**

| | Test 0 OHLCV | Test 1 MMT |
|---|---|---|
| accuracy | 51.91% | 51.32% |
| edge vs best constant | +0.84pp | +1.25pp |
| Brier (vs 0.2500) | 0.2501 | 0.2509 |
| ECE | 0.0128 | 0.0290 |
| confidence sorts accuracy | no | **yes** |

Neither beat a constant predictor by more than noise. The open question is whether
a shorter horizon or a tradeable label changes that.

**Pre-registered success bar** (decide before looking): beats its paired baseline by
**> 2pp**, holds in **both** H1 and H2, and Brier **below** the constant-predictor value.

## 0. Configuration

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ===== TEST SWITCHES =====
USE_MMT_FEATURES = True     # False = OHLCV-only baseline
USE_TAPE_SPEED   = True     # tps_* — only exists from ~2026-01-12
USE_ORDERBOOK    = True     # skew_*/depth_* — NaN before ~2026-01-10 (costs half the window)
USE_VD_COHORTS   = True     # 3 cohorts instead of 10 raw buckets

HORIZON     = 1             # minutes ahead (close-mode). Try 1, 15, 60.
LABEL_MODE  = "close"       # "close" = did close rise | "barrier" = +X before -X

# barrier-mode settings (ignored when LABEL_MODE == "close")
BARRIER_USD       = 300     # absolute $ barrier
USE_ATR_BARRIERS  = False   # True = barrier is ATR_MULT * ATR60 instead of fixed $
ATR_MULT          = 2.0
MAX_HOLD_MINUTES  = 120     # give up if neither barrier is touched

DROP_FLAT_LABELS  = True    # close-mode: drop minutes with exactly zero net move

# ===== DATA =====
SYMBOL     = "BTC"
BQ_PROJECT = "trading-brains"
BQ_TABLE   = "trading-brains.market_microstructure.mmt_btc_1m_v3"
LOOKBACK_DAYS = 360         # MMT ceiling 362 — do not raise

# ===== MODEL =====
MAX_ENCODER_LENGTH = 60
CLS_MAX_EPOCHS = 30
CLS_BATCH_SIZE = 1024       # bigger batches now that we are on GPU
CLS_LR         = 5e-4
CLS_D_MODEL    = 64
CLS_NHEAD      = 4
CLS_NUM_LAYERS = 2
CLS_DROPOUT    = 0.1
CLS_PATIENCE   = 7

import torch
USE_GPU     = torch.cuda.is_available()
ACCELERATOR = "gpu" if USE_GPU else "cpu"
PRECISION   = "16-mixed" if USE_GPU else "32-true"

RUN_TAG = (f"{SYMBOL}_{'mmt' if USE_MMT_FEATURES else 'ohlcv'}"
           f"{'_tape' if (USE_MMT_FEATURES and USE_TAPE_SPEED) else ''}"
           f"{'_nobook' if (USE_MMT_FEATURES and not USE_ORDERBOOK) else ''}"
           f"_{LABEL_MODE}"
           f"_h{HORIZON if LABEL_MODE=='close' else MAX_HOLD_MINUTES}")

print(f"Device   : {ACCELERATOR}"
      + (f"  ({torch.cuda.get_device_name(0)})" if USE_GPU else "  <-- no CUDA, see section 1"))
print(f"Precision: {PRECISION}")
print(f"Run tag  : {RUN_TAG}")
print(f"Label    : {LABEL_MODE}" + (f", horizon {HORIZON}m" if LABEL_MODE=="close"
      else f", +/-{'ATR*'+str(ATR_MULT) if USE_ATR_BARRIERS else '$'+str(BARRIER_USD)}, max hold {MAX_HOLD_MINUTES}m"))

## 1. Dependencies & GPU check

If CUDA shows **False**, your torch is a CPU-only build. Fix it in the Anaconda prompt:

```
conda activate trading_brains
pip uninstall torch -y
pip install torch --index-url https://download.pytorch.org/whl/cu126
```

then restart the kernel. (Try `cu124` if `cu126` fails.)

In [ ]:
%pip install google-cloud-storage google-cloud-bigquery db-dtypes scikit-learn

In [ ]:
import time, gc
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from google.cloud import storage, bigquery

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  device : {torch.cuda.get_device_name(0)}")
    print(f"  memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    torch.set_float32_matmul_precision("high")     # enable TF32 on Ada
else:
    print("  !! running on CPU — see the markdown cell above to enable the 4090")

## 2. Load from BigQuery

In [ ]:
def fetch_mmt_1min_data(lookback_days=360):
    client = bigquery.Client(project=BQ_PROJECT)

    tape_cols = ("tps_avg, tps_max, tps_min, tps_inst,"
                 if (USE_MMT_FEATURES and USE_TAPE_SPEED) else "")
    book_cols = ("skew_0p5pct, skew_1pct, ask_depth_1pct, bid_depth_1pct,"
                 if (USE_MMT_FEATURES and USE_ORDERBOOK) else "")
    sel_tape  = ("agg.tps_avg, agg.tps_max, agg.tps_min, agg.tps_inst,"
                 if (USE_MMT_FEATURES and USE_TAPE_SPEED) else "")
    sel_book  = ("agg.skew_0p5pct, agg.skew_1pct, agg.ask_depth_1pct, agg.bid_depth_1pct,"
                 if (USE_MMT_FEATURES and USE_ORDERBOOK) else "")

    query = f"""
    WITH bf AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT ts, open, high, low, close,
               candle_total_vol, candle_delta, candle_total_trades,
               mark_price, funding_rate,
               ROW_NUMBER() OVER (PARTITION BY ts ORDER BY ts) AS rn
        FROM `{BQ_TABLE}` WHERE exchange = 'binancef'
      ) WHERE rn = 1
    ),
    agg AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT ts, vd_b2, vd_b3, vd_b4, vd_b5, vd_b6, vd_b7, vd_b8, vd_b9,
               vd_b10, vd_b11, liq_buy_vol, liq_sell_vol, net_liq, liq_total,
               {tape_cols} {book_cols} oi_close,
               ROW_NUMBER() OVER (PARTITION BY ts ORDER BY ts) AS rn
        FROM `{BQ_TABLE}` WHERE exchange = 'binancef:bybitf'
      ) WHERE rn = 1
    )
    SELECT
      bf.ts AS timestamp, bf.open, bf.high, bf.low, bf.close,
      bf.candle_total_vol AS volume,
      bf.candle_delta, bf.candle_total_trades, bf.funding_rate, bf.mark_price,
      agg.vd_b2, agg.vd_b3, agg.vd_b4, agg.vd_b5, agg.vd_b6,
      agg.vd_b7, agg.vd_b8, agg.vd_b9, agg.vd_b10, agg.vd_b11,
      agg.liq_buy_vol, agg.liq_sell_vol, agg.net_liq, agg.liq_total,
      {sel_tape} {sel_book} agg.oi_close
    FROM bf JOIN agg USING (ts)
    WHERE bf.close > 0
      AND bf.ts >= TIMESTAMP_SUB((SELECT MAX(ts) FROM `{BQ_TABLE}`),
                                 INTERVAL {lookback_days} DAY)
    ORDER BY bf.ts
    """

    print(f"Querying BigQuery ({lookback_days}d)...")
    df = client.query(query).to_dataframe()
    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None)
    df = df.sort_values("timestamp").reset_index(drop=True)

    if USE_VD_COHORTS:
        df["vd_retail"] = df[["vd_b2", "vd_b3"]].sum(axis=1)
        df["vd_mid"]    = df[[f"vd_b{i}" for i in range(4, 10)]].sum(axis=1)
        df["vd_whale"]  = df[["vd_b10", "vd_b11"]].sum(axis=1)
        df = df.drop(columns=[f"vd_b{i}" for i in range(2, 12)])

    if not USE_MMT_FEATURES:
        df = df[["timestamp", "open", "high", "low", "close", "volume"]]

    print(f"Loaded {len(df):,} bars: {df['timestamp'].min()} -> {df['timestamp'].max()}")
    print(f"Raw columns: {len(df.columns)}")
    nn_ = df.isna().sum()
    if (nn_ > 0).any():
        print("NaN columns (these will shorten the usable window):")
        print(nn_[nn_ > 0])
    return df


df_raw = fetch_mmt_1min_data(LOOKBACK_DAYS)
df_raw.tail(3)

## 3. Features

In [ ]:
def calculate_price_features(df):
    df = df.copy()
    for p in [1, 5, 15, 30, 60]:
        df[f"returns_{p}m"] = df["close"].pct_change(p)

    df["high_low_ratio"]   = df["high"] / df["low"]
    df["high_close_ratio"] = df["high"] / df["close"]
    df["low_close_ratio"]  = df["low"] / df["close"]
    rng = (df["high"] - df["low"] + 1e-10)
    df["upper_shadow"] = (df["high"] - np.maximum(df["open"], df["close"])) / rng
    df["lower_shadow"] = (np.minimum(df["open"], df["close"]) - df["low"]) / rng

    for p in [5, 10, 20, 30]:
        sma = df["close"].rolling(p).mean()
        df[f"sma_{p}_slope"]    = sma.pct_change()
        df[f"close_to_sma_{p}"] = (df["close"] - sma) / sma
    for p in [60, 120]:
        sma = df["close"].rolling(p).mean()
        df[f"close_to_sma_{p}"] = (df["close"] - sma) / sma
    df["sma_120_slope"] = df["close"].rolling(120).mean().pct_change()

    ema12 = df["close"].ewm(span=12, adjust=False).mean()
    ema26 = df["close"].ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    df["macd_norm"] = macd / df["close"]
    df["macd_hist_norm"] = (macd - macd.ewm(span=9, adjust=False).mean()) / df["close"]

    for p in [5, 10, 20, 60]:
        df[f"volatility_{p}"] = df["returns_1m"].rolling(p).std()

    hl = df["high"] - df["low"]
    hc = (df["high"] - df["close"].shift()).abs()
    lc = (df["low"] - df["close"].shift()).abs()
    tr = np.maximum(hl, np.maximum(hc, lc))
    df["atr_60_abs"]  = tr.rolling(60).mean()          # kept for barrier sizing
    df["atr_14_norm"] = tr.rolling(14).mean() / df["close"]
    df["atr_60_norm"] = df["atr_60_abs"] / df["close"]

    for p in [20, 60]:
        sma = df["close"].rolling(p).mean()
        std = df["close"].rolling(p).std()
        df[f"bb_position_{p}"] = (df["close"] - sma) / (2 * std + 1e-10)
        df[f"bb_width_{p}"]    = std / sma

    def rsi(s, p):
        d = s.diff()
        up = d.where(d > 0, 0).rolling(p).mean()
        dn = (-d.where(d < 0, 0)).rolling(p).mean()
        return 100 - 100 / (1 + up / (dn + 1e-10))
    for p in [14, 20, 60]:
        df[f"rsi_{p}"] = rsi(df["close"], p)

    for p in [14, 60]:
        ll = df["low"].rolling(p).min(); hh = df["high"].rolling(p).max()
        df[f"stoch_k_{p}"] = 100 * (df["close"] - ll) / (hh - ll + 1e-10)
    df["stoch_d_14"] = df["stoch_k_14"].rolling(3).mean()

    df["volume_change"] = df["volume"].pct_change(1)
    for p in [5, 10, 20, 60]:
        df[f"volume_ratio_{p}"] = df["volume"] / (df["volume"].rolling(p).mean() + 1e-10)

    def mfi(d, p):
        tp = (d["high"] + d["low"] + d["close"]) / 3
        mf = tp * d["volume"]
        pos = mf.where(tp > tp.shift(), 0).rolling(p).sum()
        neg = mf.where(tp < tp.shift(), 0).rolling(p).sum()
        return 100 - 100 / (1 + pos / (neg + 1e-10))
    df["mfi_14"] = mfi(df, 14); df["mfi_60"] = mfi(df, 60)

    for p in [20, 60]:
        vw = (df["volume"] * df["close"]).rolling(p).sum() / (df["volume"].rolling(p).sum() + 1e-10)
        df[f"close_to_vwap_{p}"] = (df["close"] - vw) / vw

    sign = np.sign(df["close"].diff()).fillna(0)
    obv = (sign * df["volume"]).cumsum()
    df["obv_slope_20"] = obv.diff(20) / (df["volume"].rolling(20).sum() + 1e-10)

    h, m, d_ = df["timestamp"].dt.hour, df["timestamp"].dt.minute, df["timestamp"].dt.dayofweek
    df["hour_sin"], df["hour_cos"] = np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)
    df["minute_sin"], df["minute_cos"] = np.sin(2*np.pi*m/60), np.cos(2*np.pi*m/60)
    df["day_sin"], df["day_cos"] = np.sin(2*np.pi*d_/7), np.cos(2*np.pi*d_/7)
    return df


def calculate_flow_features(df):
    df = df.copy()
    if "oi_close" not in df.columns:
        return df
    vol = df["volume"] + 1e-10

    if USE_VD_COHORTS:
        for c in ["vd_retail", "vd_mid", "vd_whale"]:
            df[f"{c}_ratio"] = df[c] / vol
        df["vd_whale_vs_retail"] = df["vd_whale_ratio"] - df["vd_retail_ratio"]
        df = df.drop(columns=["vd_retail", "vd_mid", "vd_whale"])
    else:
        for i in range(2, 12):
            df[f"vd_b{i}_ratio"] = df[f"vd_b{i}"] / vol
        df = df.drop(columns=[f"vd_b{i}" for i in range(2, 12)])

    df["candle_delta_ratio"] = df["candle_delta"] / vol
    ats = df["volume"] / (df["candle_total_trades"] + 1e-10)
    df["avg_trade_size_ratio"] = ats / (ats.rolling(60).mean() + 1e-10)

    df["liq_intensity"] = df["liq_total"] / (df["liq_total"].rolling(240).mean() + 1e-10)
    df["liq_imbalance"] = df["net_liq"] / (df["liq_total"] + 1e-10)
    df["liq_vs_volume"] = df["liq_total"] / vol

    df["oi_change_1m"]  = df["oi_close"].pct_change(1)
    df["oi_change_15m"] = df["oi_close"].pct_change(15)
    oi_sma = df["oi_close"].rolling(240).mean()
    df["oi_vs_sma"] = (df["oi_close"] - oi_sma) / (oi_sma + 1e-10)

    if "ask_depth_1pct" in df.columns:
        tot = df["ask_depth_1pct"] + df["bid_depth_1pct"] + 1e-10
        df["depth_imbalance"] = (df["bid_depth_1pct"] - df["ask_depth_1pct"]) / tot
        df["depth_vs_sma"]    = tot / (tot.rolling(240).mean() + 1e-10)
        df = df.drop(columns=["ask_depth_1pct", "bid_depth_1pct"])

    if "tps_avg" in df.columns:
        # tape speed as ratios to its own baseline, never raw counts
        df["tps_avg_ratio"]  = df["tps_avg"] / (df["tps_avg"].rolling(240).mean() + 1e-10)
        df["tps_burst"]      = df["tps_max"] / (df["tps_avg"] + 1e-10)
        df["tps_range"]      = (df["tps_max"] - df["tps_min"]) / (df["tps_avg"] + 1e-10)
        df["tps_inst_ratio"] = df["tps_inst"] / (df["tps_avg"] + 1e-10)
        df = df.drop(columns=["tps_avg", "tps_max", "tps_min", "tps_inst"])

    fr = df["funding_rate"]
    df["funding_z"] = (fr - fr.rolling(480).mean()) / (fr.rolling(480).std() + 1e-10)
    df["mark_dislocation_bps"] = (df["close"] - df["mark_price"]) / df["close"] * 1e4

    df = df.drop(columns=[c for c in ["candle_delta", "candle_total_trades",
                                      "liq_buy_vol", "liq_sell_vol", "net_liq",
                                      "liq_total", "oi_close", "mark_price"]
                          if c in df.columns])
    return df

### 3b. Labels

`close` mode: did close rise over `HORIZON` minutes.

`barrier` mode: walk forward bar by bar and record which barrier is touched **first**
(+BARRIER before −BARRIER). This is path-based, not endpoint-based — it uses each
bar's high/low, so it reflects what a real bracket order would actually experience.
Unresolved within `MAX_HOLD_MINUTES` are dropped.

In [ ]:
def make_close_labels(df):
    fut = df["close"].shift(-HORIZON)
    lab = (fut > df["close"]).astype("float32")
    lab[fut.isna()] = np.nan
    if DROP_FLAT_LABELS:
        lab[(fut == df["close"])] = np.nan          # exact ties carry no information
    df["target"] = lab
    return df


def make_barrier_labels(df):
    """First-touch labelling. 1 = upper barrier hit first, 0 = lower first, NaN = neither."""
    close = df["close"].to_numpy()
    high  = df["high"].to_numpy()
    low   = df["low"].to_numpy()
    n = len(df)

    if USE_ATR_BARRIERS:
        bar = (df["atr_60_abs"] * ATR_MULT).to_numpy()
    else:
        bar = np.full(n, float(BARRIER_USD))

    out = np.full(n, np.nan, dtype="float32")
    H = MAX_HOLD_MINUTES
    for i in range(n - 1):
        b = bar[i]
        if not np.isfinite(b) or b <= 0:
            continue
        up, dn = close[i] + b, close[i] - b
        end = min(i + H, n - 1)
        for j in range(i + 1, end + 1):
            hit_up = high[j] >= up
            hit_dn = low[j] <= dn
            if hit_up and hit_dn:
                break                      # ambiguous bar: unknowable order, discard
            if hit_up:
                out[i] = 1.0; break
            if hit_dn:
                out[i] = 0.0; break
    df["target"] = out
    return df


def prepare_dataset(df):
    df = calculate_price_features(df)
    df = calculate_flow_features(df)

    if LABEL_MODE == "barrier":
        print(f"Building barrier labels (this walks {len(df):,} bars — takes a minute)...")
        t0 = time.time()
        df = make_barrier_labels(df)
        print(f"  done in {time.time()-t0:.0f}s")
        res = df["target"].notna().mean()
        print(f"  resolved within {MAX_HOLD_MINUTES}m: {res:.1%} of bars")
    else:
        df = make_close_labels(df)

    df = df.replace([np.inf, -np.inf], np.nan)
    feat_cols = [c for c in df.columns if c not in ("timestamp", "target")]
    df = df.dropna(subset=feat_cols).dropna(subset=["target"]).reset_index(drop=True)

    # correlation prune
    exclude = {"timestamp", "target", "open", "high", "low", "close", "volume", "atr_60_abs"}
    feats = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
    corr = df[feats].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop = [c for c in upper.columns if any(upper[c] > 0.95)]
    df = df.drop(columns=drop)
    print(f"Correlation filter: {len(feats)} -> {len(feats)-len(drop)} (dropped {len(drop)})")

    print(f"Prepared: {len(df):,} rows, {len(df.columns)} columns")
    return df


df_all = prepare_dataset(df_raw)
data_start, data_end = str(df_all.timestamp.min()), str(df_all.timestamp.max())
print(f"Range: {data_start} -> {data_end}")
print(f"Base rate (share of 1-labels): {df_all['target'].mean():.2%}")
del df_raw; gc.collect()

## 4. Split + scale

In [ ]:
EXCLUDE = ["timestamp", "target", "open", "high", "low", "close", "volume", "atr_60_abs"]
feature_cols = [c for c in df_all.columns
                if c not in EXCLUDE and pd.api.types.is_numeric_dtype(df_all[c])]
flow_cols = [c for c in feature_cols if any(k in c for k in
             ("vd_", "liq_", "oi_", "depth", "funding", "mark_disl",
              "candle_delta", "avg_trade_size", "tps_"))]
print(f"Features: {len(feature_cols)}  (price {len(feature_cols)-len(flow_cols)} | flow {len(flow_cols)})")
if flow_cols:
    print(f"  flow -> {flow_cols}")

cutoff = int(len(df_all) * 0.8)
df_train, df_val = df_all.iloc[:cutoff], df_all.iloc[cutoff:]
print(f"\nTrain {len(df_train):,}  {df_train.timestamp.min().date()} -> {df_train.timestamp.max().date()}")
print(f"Val   {len(df_val):,}  {df_val.timestamp.min().date()} -> {df_val.timestamp.max().date()}")
print(f"Base rate  train {df_train.target.mean():.2%} | val {df_val.target.mean():.2%}")

scaler = StandardScaler().fit(df_train[feature_cols].values)
Xtr = scaler.transform(df_train[feature_cols].values).astype("float32")
Xva = scaler.transform(df_val[feature_cols].values).astype("float32")
ytr = df_train["target"].values.astype("float32")
yva = df_val["target"].values.astype("float32")
print("\nScaler fitted on train only.")

## 5. Fast GPU dataset

The old dataset sliced a numpy window per sample in Python — that made the CPU the
bottleneck and left the GPU idle. Here the whole feature matrix lives on the GPU as
one tensor (~60 MB) and each batch is gathered with a single indexing op. Typically
10-50x faster.

In [ ]:
DEV = torch.device("cuda" if USE_GPU else "cpu")

class WindowBank:
    """Holds features/labels on-device; gathers (B, enc, F) windows by index."""
    def __init__(self, X, y, enc):
        self.X = torch.from_numpy(X).to(DEV)
        self.y = torch.from_numpy(y).to(DEV)
        self.enc = enc
        self.n = len(X) - enc + 1
        self.offsets = torch.arange(enc, device=DEV)

    def gather(self, idx):
        rows = idx.unsqueeze(1) + self.offsets.unsqueeze(0)   # (B, enc)
        return self.X[rows], self.y[idx + self.enc - 1]


class IndexDataset(Dataset):
    def __init__(self, n): self.n = n
    def __len__(self): return self.n
    def __getitem__(self, i): return i


class DirectionClassifier(pl.LightningModule):
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2,
                 dropout=0.1, learning_rate=5e-4, focal_alpha=0.5, focal_gamma=1.5):
        super().__init__()
        self.save_hyperparameters()
        self.bank_train = None; self.bank_val = None
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, 256, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                    dim_feedforward=d_model*4, dropout=dropout,
                    batch_first=True, activation="gelu")
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model//2),
                                  nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model//2, 1))

    def forward(self, x):
        h = self.input_proj(x)
        h = h + self.pos_embedding[:, :h.size(1), :]
        h = self.transformer(h)
        return self.head(h[:, -1, :]).squeeze(-1)

    def focal_loss(self, logits, y):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, y, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p*y + (1-p)*(1-y)
        a_t = self.hparams.focal_alpha*y + (1-self.hparams.focal_alpha)*(1-y)
        return (a_t * (1-p_t)**self.hparams.focal_gamma * bce).mean()

    def _step(self, idx, bank, stage):
        x, y = bank.gather(idx)
        logits = self(x)
        loss = self.focal_loss(logits, y)
        acc = ((torch.sigmoid(logits) > 0.5).float() == y).float().mean()
        self.log(f"{stage}_loss", loss, prog_bar=True, batch_size=len(idx))
        self.log(f"{stage}_acc", acc, prog_bar=True, batch_size=len(idx))
        return loss

    def training_step(self, idx, _):   return self._step(idx, self.bank_train, "train")
    def validation_step(self, idx, _): return self._step(idx, self.bank_val, "val")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=0.01)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.trainer.max_epochs)
        return [opt], [sch]


bank_train = WindowBank(Xtr, ytr, MAX_ENCODER_LENGTH)
bank_val   = WindowBank(Xva, yva, MAX_ENCODER_LENGTH)
print(f"Train windows {bank_train.n:,} | Val windows {bank_val.n:,}")

collate = lambda b: torch.tensor(b, device=DEV)
train_dl = DataLoader(IndexDataset(bank_train.n), batch_size=CLS_BATCH_SIZE,
                      shuffle=True, collate_fn=collate, num_workers=0, drop_last=True)
val_dl   = DataLoader(IndexDataset(bank_val.n), batch_size=CLS_BATCH_SIZE,
                      shuffle=False, collate_fn=collate, num_workers=0)

## 6. Train

In [ ]:
model = DirectionClassifier(n_features=len(feature_cols), d_model=CLS_D_MODEL,
                            nhead=CLS_NHEAD, num_layers=CLS_NUM_LAYERS,
                            dropout=CLS_DROPOUT, learning_rate=CLS_LR)
model.bank_train, model.bank_val = bank_train, bank_val
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e3:.1f}k")

ckpt_dir = f"checkpoints/{RUN_TAG}"
os.makedirs(ckpt_dir, exist_ok=True)
ckpt = ModelCheckpoint(dirpath=ckpt_dir, monitor="val_loss",
                       filename="cls-{epoch:02d}-{val_loss:.4f}", save_top_k=1, mode="min")

trainer = pl.Trainer(max_epochs=CLS_MAX_EPOCHS, accelerator=ACCELERATOR,
                     devices=1, precision=PRECISION, gradient_clip_val=1.0,
                     callbacks=[EarlyStopping(monitor="val_loss", patience=CLS_PATIENCE,
                                              mode="min", verbose=True),
                                ckpt, LearningRateMonitor(logging_interval="epoch")],
                     enable_model_summary=True, log_every_n_steps=50)

t0 = time.time()
print(f"Training {RUN_TAG} on {ACCELERATOR} ({PRECISION})...")
trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl)
print(f"Wall clock: {(time.time()-t0)/60:.1f} min")

## 7. Evaluate

In [ ]:
best = DirectionClassifier.load_from_checkpoint(ckpt.best_model_path,
                                                n_features=len(feature_cols))
best.bank_val = bank_val
best.eval().to(DEV)

P, L = [], []
with torch.no_grad():
    for idx in val_dl:
        x, y = bank_val.gather(idx)
        P.append(torch.sigmoid(best(x)).float().cpu()); L.append(y.float().cpu())
probs = torch.cat(P).numpy(); labels = torch.cat(L).numpy()

base = labels.mean()
best_const = max(base, 1 - base)                 # "always up" or "always down"
pred = (probs > 0.5).astype(int)
acc = (pred == labels).mean()
brier = float(np.mean((probs - labels) ** 2))
brier_const = float(np.mean((base - labels) ** 2))
ece = 0.0
for lo, hi in zip(np.linspace(0,1,11)[:-1], np.linspace(0,1,11)[1:]):
    m = (probs >= lo) & (probs < hi)
    if m.sum(): ece += m.mean() * abs(probs[m].mean() - labels[m].mean())

# effective sample size: overlapping windows are not independent
overlap = HORIZON if LABEL_MODE == "close" else MAX_HOLD_MINUTES
n_eff = max(1, len(labels) // max(1, overlap))
se = np.sqrt(0.25 / n_eff) * 100
edge = (acc - best_const) * 100

print("=" * 66)
print(f"RESULTS — {RUN_TAG}")
print("=" * 66)
print(f"Validation windows : {len(labels):,}   (effective independent ~{n_eff:,})")
print(f"Base rate          : {base*100:.2f}%   best constant = {best_const*100:.2f}%")
print(f"Accuracy           : {acc*100:.2f}%")
print(f"EDGE vs constant   : {edge:+.2f} pp   (1 s.e. ~ {se:.2f} pp -> {abs(edge)/se:.1f} sigma)")
print(f"Brier              : {brier:.4f}   (constant = {brier_const:.4f}) "
      f"{'BETTER' if brier < brier_const else 'worse'}")
print(f"ECE                : {ece:.4f}")
print(f"Predicted UP share : {pred.mean()*100:.1f}%")

print("\nCalibration:")
print(f"  {'bucket':<14}{'n':>9}{'mean p':>10}{'actual':>10}{'gap':>8}")
for lo, hi in [(0,.3),(.3,.4),(.4,.45),(.45,.5),(.5,.55),(.55,.6),(.6,.7),(.7,1.)]:
    m = (probs >= lo) & (probs < hi)
    if m.sum() < 20: continue
    print(f"  [{lo:.2f},{hi:.2f}){m.sum():>9,}{probs[m].mean():>10.3f}"
          f"{labels[m].mean()*100:>9.1f}%{(labels[m].mean()-probs[m].mean())*100:>8.1f}")

half = len(labels)//2
print("\nRegime stability:")
for nm, sl in [("H1", slice(0,half)), ("H2", slice(half,None))]:
    p_, l_ = probs[sl], labels[sl]
    bc = max(l_.mean(), 1-l_.mean())
    print(f"  {nm}: n={len(l_):,}  base={l_.mean()*100:.1f}%  "
          f"acc={((p_>0.5)==l_).mean()*100:.1f}%  edge={(((p_>0.5)==l_).mean()-bc)*100:+.2f}pp")

print("\nAccuracy by confidence:")
conf = np.abs(probs - 0.5)
for lo, hi in [(0,.02),(.02,.05),(.05,.1),(.1,.2),(.2,.5)]:
    m = (conf >= lo) & (conf < hi)
    if m.sum() < 20: continue
    ne = max(1, m.sum()//overlap)
    print(f"  conf [{lo:.2f},{hi:.2f}): n={m.sum():>8,} (~{ne:>5,} indep)  "
          f"acc={((probs[m]>0.5)==labels[m]).mean()*100:.1f}%  +/-{np.sqrt(0.25/ne)*100*1.96:.1f}pp")
print("  ^ accuracy should RISE with confidence.")

print("\nPRE-REGISTERED BAR: edge > 2pp AND positive in both halves AND Brier < constant.")
results = dict(run_tag=RUN_TAG, accuracy=float(acc), base_rate=float(base),
               best_const=float(best_const), edge_pp=float(edge), sigma=float(abs(edge)/se),
               brier=brier, brier_const=brier_const, ece=float(ece),
               n_features=len(feature_cols), n_val=int(len(labels)), n_eff=int(n_eff))
print(f"\n>>> {results}")

## 8. Test log

Paste each run's `results` dict here as you go.

In [ ]:
ALL_RUNS = [
    dict(run_tag="BTC_ohlcv_close_h60", accuracy=.5191, best_const=.5107, edge_pp=+0.84,
         brier=.2501, brier_const=.2499, ece=.0128, n_features=53),
    dict(run_tag="BTC_mmt_close_h60",   accuracy=.5132, best_const=.5007, edge_pp=+1.25,
         brier=.2509, brier_const=.2500, ece=.0290, n_features=69),
    # append new runs here
]
print(f"{'run':<34}{'acc':>8}{'edge pp':>9}{'brier':>9}{'vs const':>10}{'ece':>8}")
print("-" * 78)
for r in ALL_RUNS:
    flag = "BETTER" if r["brier"] < r["brier_const"] else "worse"
    print(f"{r['run_tag']:<34}{r['accuracy']*100:>7.2f}%{r['edge_pp']:>+9.2f}"
          f"{r['brier']:>9.4f}{flag:>10}{r['ece']:>8.4f}")